# Qwen2.5-1.5B-Instruct QLoRA Training on 10K Augmented Samples

This notebook:
1. Builds a 10K dataset using augment_queries.py from the base trajectories file.
2. Converts data to SFT chat format for Qwen2.5-1.5B-Instruct.
3. Trains with QLoRA on a single NVIDIA GPU.
4. Saves adapter checkpoints, tokenizer, trainer state, and merged model weights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies (Colab/local)
%pip install -q -U transformers datasets peft accelerate bitsandbytes sentencepiece

In [ ]:
import json
import os
import random
import subprocess
from datetime import datetime
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
 )

set_seed(42)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA capability:', torch.cuda.get_device_capability(0))
    print('BF16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
# Colab setup (safe to run locally too)
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
print('IN_COLAB:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Put your uploaded project folder here in Drive.
    # Example target path after upload: /content/drive/MyDrive/dl_project/qwen-claude
    COLAB_PROJECT_DIR = Path('/content/drive/MyDrive/dl_project/qwen-claude')
    if not COLAB_PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project folder not found: {COLAB_PROJECT_DIR}. '
            'Upload/copy your project to Drive and update COLAB_PROJECT_DIR if needed.'
        )
    os.chdir(str(COLAB_PROJECT_DIR))
    print('Working directory set to:', Path.cwd())

In [ ]:
# Paths and run configuration
ROOT = Path.cwd()
TARGET_SAMPLES = 10000
BASE_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

# Resolve project root for both local and Colab layouts.
if (ROOT / 'augment_queries.py').exists():
    PROJECT_ROOT = ROOT
elif (ROOT / 'qwen-claude' / 'augment_queries.py').exists():
    PROJECT_ROOT = ROOT / 'qwen-claude'
else:
    raise FileNotFoundError('Could not locate augment_queries.py from current working directory.')

# Try common input locations automatically
input_candidates = [
    PROJECT_ROOT / 'agent_trajectories.json',
    PROJECT_ROOT / 'agent_trajectories_2k.json',
    ROOT / 'agent_trajectories.json',
    ROOT / 'agent_trajectories_2k.json',
]

INPUT_JSON = next((p for p in input_candidates if p.exists()), None)
if INPUT_JSON is None:
    raise FileNotFoundError('Could not find agent_trajectories.json or agent_trajectories_2k.json in expected locations.')

RUN_ROOT = PROJECT_ROOT / 'training_runs' / ('qwen2.5-1.5b-colab-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
DATA_DIR = RUN_ROOT / 'data'
OUTPUT_DIR = RUN_ROOT / 'adapter'
MERGED_DIR = RUN_ROOT / 'merged_model'

AUGMENT_SCRIPT = PROJECT_ROOT / 'augment_queries.py'
AUG_JSON = DATA_DIR / 'augmented_target.json'
TRAIN_JSONL = DATA_DIR / 'train_sft.jsonl'

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('Input JSON:', INPUT_JSON)
print('Run root:', RUN_ROOT)
print('Augment script:', AUGMENT_SCRIPT)
print('Augmented dataset path:', AUG_JSON)
print('Train JSONL path:', TRAIN_JSONL)
print('Adapter output path:', OUTPUT_DIR)

In [ ]:
# Step 1: Generate augmented dataset to target size using augment_queries.py
cmd = [
    'python', str(AUGMENT_SCRIPT),
    '--input_json', str(INPUT_JSON),
    '--output_json', str(AUG_JSON),
    '--target_count', str(TARGET_SAMPLES),
    '--seed', '42',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

with AUG_JSON.open('r', encoding='utf-8') as f:
    samples = json.load(f)

# Ensure exact 10K rows even if dedup removed some entries
if len(samples) < TARGET_SAMPLES:
    rng = random.Random(42)
    needed = TARGET_SAMPLES - len(samples)
    samples.extend(rng.choices(samples, k=needed))
    rng.shuffle(samples)
elif len(samples) > TARGET_SAMPLES:
    rng = random.Random(42)
    rng.shuffle(samples)
    samples = samples[:TARGET_SAMPLES]

with AUG_JSON.open('w', encoding='utf-8') as f:
    json.dump(samples, f, ensure_ascii=False, indent=2)

print('Final augmented sample count:', len(samples))

In [ ]:
# Step 2: Convert (query, actions) into chat-style SFT examples
SYSTEM_PROMPT = (
    'You are a data-analysis tool-planning agent. '
    'Return ONLY valid JSON with keys: actions, answer. '
    'actions must be a list of tool calls as strings, and answer must be null.'
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_prompt_text(query: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Query: {query}\nOutput JSON:'},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def make_target_text(actions: list[str]) -> str:
    assistant_obj = {'actions': actions, 'answer': None}
    # Add EOS so the model learns where valid JSON completion should stop.
    return json.dumps(assistant_obj, ensure_ascii=False) + tokenizer.eos_token

valid_rows = 0
with TRAIN_JSONL.open('w', encoding='utf-8') as out_f:
    for row in samples:
        if not isinstance(row, dict):
            continue
        q = row.get('query')
        a = row.get('actions')
        if not q or not isinstance(a, list) or not all(isinstance(x, str) for x in a):
            continue

        prompt_text = make_prompt_text(q)
        target_text = make_target_text(a)
        full_text = prompt_text + target_text

        out_f.write(
            json.dumps(
                {
                    'prompt_text': prompt_text,
                    'target_text': target_text,
                    'text': full_text,
                },
                ensure_ascii=False,
            ) + '\n'
        )
        valid_rows += 1

print('Wrote SFT rows:', valid_rows)
print('Training file:', TRAIN_JSONL)

In [ ]:
# Step 3: Load dataset and model for QLoRA
import gc

train_ds = load_dataset('json', data_files={'train': str(TRAIN_JSONL)})['train']
print(train_ds)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

# Clear cached allocations from previously run cells to maximize free VRAM.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,

 )

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected. QLoRA training in this notebook requires an NVIDIA GPU.')

# Force model to stay on GPU 0; avoids auto device_map sending modules to CPU/disk.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation="eager"
 )

model.config.use_cache = False
# model.gradient_checkpointing_enable(
#     gradient_checkpointing_kwargs={"use_reentrant": False}
# )
# model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # set here, not before
)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 4: Train QLoRA adapter (completion-only objective for JSON output)
# Tune these for your VRAM; defaults are laptop-friendly for many 8-12 GB GPUs.
per_device_batch_size = 2
grad_accum = 8
num_epochs = 2
max_seq_len = 768
learning_rate = 2e-4

tokenizer.padding_side = 'right'

def tokenize_batch(batch):
    prompt_ids_batch = tokenizer(
        batch['prompt_text'],
        truncation=True,
        max_length=max_seq_len,
        padding=False,
        add_special_tokens=False,
    )['input_ids']

    full_ids_batch = tokenizer(
        batch['text'],
        truncation=True,
        max_length=max_seq_len,
        padding=False,
        add_special_tokens=False,
    )['input_ids']

    out_input_ids = []
    out_attention = []
    out_labels = []

    for p_ids, f_ids in zip(prompt_ids_batch, full_ids_batch):
        labels = f_ids.copy()
        cutoff = min(len(p_ids), len(labels))
        labels[:cutoff] = [-100] * cutoff

        out_input_ids.append(f_ids)
        out_attention.append([1] * len(f_ids))
        out_labels.append(labels)

    return {
        'input_ids': out_input_ids,
        'attention_mask': out_attention,
        'labels': out_labels,
    }

tokenized_train = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_ds.column_names,
    desc='Tokenizing train set',
)

# Pads input_ids/attention_mask and labels (with -100) correctly for variable-length batches.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

steps_per_epoch = max(1, len(tokenized_train) // (per_device_batch_size * grad_accum))
warmup_steps = max(10, int(0.03 * steps_per_epoch * num_epochs))

train_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=per_device_batch_size,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=num_epochs,
    learning_rate=learning_rate,
    lr_scheduler_type='cosine',
    warmup_steps=warmup_steps,
    logging_steps=20,
    save_steps=200,
    save_total_limit=3,
    bf16=use_bf16,
    fp16=not use_bf16,
    optim='paged_adamw_8bit',
    report_to='none',
    remove_unused_columns=False,
    gradient_checkpointing=True,                              # ← uncomment
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

train_result = trainer.train()
print(train_result)

In [ ]:
# Step 5: Save adapter, tokenizer, and run metadata
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
trainer.save_state()

meta = {
    'base_model': BASE_MODEL_ID,
    'target_samples': TARGET_SAMPLES,
    'actual_samples': len(samples),
    'train_rows': len(train_ds),
    'output_dir': str(OUTPUT_DIR),
    'merged_dir': str(MERGED_DIR),
}
with (RUN_ROOT / 'run_metadata.json').open('w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print('Adapter and tokenizer saved to:', OUTPUT_DIR)
print('Run metadata saved to:', RUN_ROOT / 'run_metadata.json')

In [ ]:
# Step 6 (Optional): Merge LoRA adapter into base model and save full weights
# This needs much more RAM/VRAM and disk than adapter-only saving.
do_merge = False

if do_merge:
    base_fp16 = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )

    peft_model = PeftModel.from_pretrained(base_fp16, str(OUTPUT_DIR))
    merged_model = peft_model.merge_and_unload()

    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)
    print('Merged model saved to:', MERGED_DIR)
else:
    print('Skipped merge step.')

In [ ]:
# Quick sanity-generation check using the adapter
prompt = 'Query: Top 3 cities by revenue in 2022\nOutput JSON:'
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': prompt},
]
inference_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

test_model = trainer.model
test_model.eval()
if hasattr(test_model, 'gradient_checkpointing_disable'):
    test_model.gradient_checkpointing_disable()
test_model.config.use_cache = True

# Silence greedy-decoding warnings from inherited generation config.
test_model.generation_config.do_sample = False
test_model.generation_config.temperature = None
test_model.generation_config.top_p = None
test_model.generation_config.top_k = None

inputs = tokenizer(inference_prompt, return_tensors='pt').to(test_model.device)

with torch.no_grad():
    out = test_model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

gen_ids = out[0][inputs['input_ids'].shape[1]:]
gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
print('Raw generation:\n', gen_text)

start = gen_text.find('{')
end = gen_text.rfind('}')
if start != -1 and end != -1 and end > start:
    candidate = gen_text[start:end+1]
    try:
        parsed = json.loads(candidate)
        print('\nParsed JSON OK:')
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
    except Exception as e:
        print('\nJSON parse failed:', repr(e))
else:
    print('\nNo complete JSON object found in generation.')

In [ ]:
# Evaluate saved adapter on test_trajectory_2k.json (NO TRAINING)
from pathlib import Path
import json
import re
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

# Resolve paths
if 'PROJECT_ROOT' in globals():
    eval_project_root = PROJECT_ROOT
else:
    cwd = Path.cwd()
    if (cwd / 'augment_queries.py').exists():
        eval_project_root = cwd
    elif (cwd / 'qwen-claude' / 'augment_queries.py').exists():
        eval_project_root = cwd / 'qwen-claude'
    else:
        raise FileNotFoundError('Could not resolve qwen-claude project root.')

test_candidates = [
    eval_project_root / 'test_trajectory_2k.json',
    eval_project_root.parent / 'test_trajectory_2k.json',
]
test_path = next((p for p in test_candidates if p.exists()), None)
if test_path is None:
    raise FileNotFoundError('Could not find test_trajectory_2k.json in expected locations.')

if 'OUTPUT_DIR' not in globals():
    raise RuntimeError('OUTPUT_DIR not found. Run the setup/saving cells first.')
adapter_dir = Path(OUTPUT_DIR)
if not adapter_dir.exists():
    raise FileNotFoundError(f'Adapter directory does not exist: {adapter_dir}')

# Load tokenizer from adapter directory
eval_tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir), use_fast=True)
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

# Load PEFT model directly from saved adapter (avoids PeftModel+device_map bug).
if torch.cuda.is_available():
    eval_model = AutoPeftModelForCausalLM.from_pretrained(
        str(adapter_dir),
        torch_dtype=torch.float16,
        device_map={'': 0},
        trust_remote_code=True,
    )
else:
    eval_model = AutoPeftModelForCausalLM.from_pretrained(
        str(adapter_dir),
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

eval_model.eval()
if hasattr(eval_model, 'gradient_checkpointing_disable'):
    eval_model.gradient_checkpointing_disable()
eval_model.config.use_cache = True

def extract_actions_from_text(gen_text: str):
    start = gen_text.find('{')
    end = gen_text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return None, 'no-json-found'
    candidate = gen_text[start:end+1]
    try:
        obj = json.loads(candidate)
    except Exception:
        return None, 'json-parse-failed'
    actions = obj.get('actions')
    if not isinstance(actions, list) or not all(isinstance(a, str) for a in actions):
        return None, 'invalid-actions-field'
    return actions, None

def norm_action(a: str) -> str:
    # Normalize whitespace for fair string comparison while preserving function syntax.
    return re.sub(r'\s+', '', a.strip())

with test_path.open('r', encoding='utf-8') as f:
    test_rows = json.load(f)

valid_rows = [r for r in test_rows if isinstance(r, dict) and isinstance(r.get('query'), str) and isinstance(r.get('actions'), list)]
total = len(valid_rows)
correct = 0
parse_failures = 0
mismatches = []

for i, row in enumerate(valid_rows, start=1):
    q = row['query']
    gold_actions = row['actions']

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Query: {q}\nOutput JSON:'},
    ]
    prompt_text = eval_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = eval_tokenizer(prompt_text, return_tensors='pt').to(eval_model.device)

    with torch.no_grad():
        out_ids = eval_model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=eval_tokenizer.eos_token_id,
            pad_token_id=eval_tokenizer.eos_token_id,
        )

    gen_only = out_ids[0][inputs['input_ids'].shape[1]:]
    gen_text = eval_tokenizer.decode(gen_only, skip_special_tokens=True).strip()
    pred_actions, err = extract_actions_from_text(gen_text)

    if err is not None:
        parse_failures += 1
        mismatches.append({
            'idx': i - 1,
            'query': q,
            'reason': err,
            'gold': gold_actions,
            'pred': gen_text[:300],
        })
        running_acc = correct / i
        print(f'TC {i}/{total} | status=parse-fail ({err}) | running acc={running_acc:.4f} ({running_acc*100:.2f}%)')
        continue

    gold_norm = [norm_action(a) for a in gold_actions]
    pred_norm = [norm_action(a) for a in pred_actions]

    if pred_norm == gold_norm:
        correct += 1
        status = 'match'
    else:
        status = 'mismatch'
        mismatches.append({
            'idx': i - 1,
            'query': q,
            'reason': 'actions-mismatch',
            'gold': gold_actions,
            'pred': pred_actions,
        })

    running_acc = correct / i
    print(f'TC {i}/{total} | status={status} | running acc={running_acc:.4f} ({running_acc*100:.2f}%)')

acc = (correct / total) if total else 0.0
print(f'\nTest file: {test_path}')
print(f'Adapter dir: {adapter_dir}')
print(f'Total evaluated: {total}')
print(f'Exact action-match correct: {correct}')
print(f'Exact match accuracy: {acc:.4f} ({acc*100:.2f}%)')
print(f'JSON parse failures: {parse_failures}')

# Show a few error examples
show_n = min(5, len(mismatches))
if show_n > 0:
    print(f'\nShowing {show_n} mismatch examples:')
    for m in mismatches[:show_n]:
        print('-' * 90)
        print('idx:', m['idx'])
        print('reason:', m['reason'])
        print('query:', m['query'])
        print('gold:', m['gold'])
        print('pred:', m['pred'])
else:
    print('\nAll evaluated queries matched exactly.')